In [0]:
%python
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim

spark = SparkSession.builder.getOrCreate()

BRONZE_TABLE = "sales_tax_catalog.bronze.sales_tax_raw"
SILVER_TABLE = "sales_tax_catalog.silver.sales_tax_clean"

def clean_silver():
    df_raw = spark.table(BRONZE_TABLE)
    
    # Limpeza e padronização das colunas
    df_clean = df_raw.select(
        trim(col("City")).alias("city_name"),
        col("`Report Month`").cast("int").alias("report_month"),
        col("`Report Year`").cast("int").alias("report_year"),
        col("`Report Period Type`").alias("report_period_type"),
        col("`Net Payment This Period`").cast("double").alias("net_payment"),
        col("`Payment to Date`").cast("double").alias("payment_to_date")
    ).dropna(subset=["city_name", "net_payment"])
    
    # Gravando na Silver com Change Data Feed ativado
    df_clean.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .option("delta.enableChangeDataFeed", "true") \
        .saveAsTable(SILVER_TABLE)
        
    print(f"Camada Silver processada com sucesso: {SILVER_TABLE}")

if __name__ == "__main__":
    clean_silver()